## Organized reproduction copy

Run the setup cell first, then Run All. Original analysis cells and saved UCLA outputs below are unchanged. The setup only resolves the organized folder paths. Inputs are linked from `04_results/reproduction_inputs`; new runs write to `04_results/reruns`, preserving the supplied results. Notebook 01 reconstructs the feature table; notebooks 02–03 use the included validated table. Older mapping-unverified wording is historical: all 2,661 mappings were subsequently verified in notebook 03.


In [ ]:
# Folder setup only; the original analysis cells below are unchanged.
from pathlib import Path
import os
_here = Path.cwd().resolve()
_project = next((p for p in [_here, *_here.parents] if (p/'01_original_data').is_dir() and (p/'02_notebooks').is_dir() and (p/'04_results').is_dir()), None)
if _project is None:
    raise FileNotFoundError('Start Jupyter inside the organized wallet_profit_research folder.')
os.chdir(_project/'02_notebooks')
os.environ['WALLET_MODEL_ROOT'] = str(Path.cwd())
os.environ['WALLET_VALIDATION_ROOT'] = str(Path.cwd())
os.environ['WALLET_MODEL_MODE'] = 'production'
os.environ['WALLET_VALIDATION_MODE'] = 'production'
os.environ['WALLET_MODEL_OUTPUT'] = str(_project/'04_results/reruns/modeling/production')
os.environ['WALLET_VALIDATION_OUTPUT'] = str(_project/'04_results/reruns/validation/production')
assert (Path('data')/'wallet_market_candidates_full_week.parquet').is_file()
assert len(list((Path('data')/'weekly_trades').glob('trades-*.parquet'))) == 7
print('Inputs:', Path('data').resolve())
print('New outputs:', (_project/'04_results/reruns').resolve())


# 钱包条件盈利：独立关联与样本外预测

本轮实际拟合模型，分别回答：① 同一市场内、控制其他预先规定变量后，某项特征与条件毛利为正是否有关联；② 在时间更晚、事件组不同的测试样本中，加入/删去某类特征能否改善概率预测。统计显著不等于预测有增益，未显著也不等于无价值，更不是因果证据。

标签是**假设下的费用前观察毛利**，不是已确认净利润。候选市场宇宙已经用事后开始/结算及交易覆盖条件筛选；下面的时间验证仍是该事后宇宙内的回顾性评估。记录结算时间只是结果可用性的代理：即使预测早于记录结算，结果仍可能已公开。更好的数据要求：正确token/结果映射、真实费用与期初成本、非交易持仓变化、结果公开/停盘历史、核验事件ID和更长时间跨度。

本笔记本读取上一轮经逐笔核验的候选表：每个钱包–市场只取一次预测，时间为提供的一周内首次观察时间戳的整批成交完成后，并非首次终身入场。表包含该时点当前特征、严格更早的同周历史、最终会计诊断和未核验标签；原始会计在上一轮自包含笔记本重现。

标签公式：`gross PNL = sell dollars − buy dollars + net shares[0] × payout[0] + net shares[1] × payout[1]`，`net shares = BUY size − SELL size`。主场景假设 `payout[0]=price_at_close`、`payout[1]=1−price_at_close`，映射**未独立核验**，同时假设零期初持仓、完整交易账本、无未记持仓变化，且未扣手续费。`Y=1[PNL>1e−6]`；持平及非正毛利归0。出现负观察库存的组合不提供条件标签，保留为缺失；全部 confirmed net 标签仍是NA。反转映射/去重只是敏感性场景。

49列全部登记，但代数别名、恒定列、极罕见卖出和双边买入模式没有虚构的独立p值。当前价格是买入合约均价，不是跨所有市场统一的Yes概率。过去历史只涵盖提供的一周，不是终身经验或已证明的全球账本。

重现：`WALLET_MODEL_MODE=production python execute_modeling_notebook.py`。默认核 `walletmodel`；可用 `WALLET_MODEL_KERNEL` 覆盖。输入位于 `data/`，输出 `results/modeling/production/`；可通过 `WALLET_MODEL_ROOT` / `WALLET_MODEL_OUTPUT` 改路径。`smoke` 使用同一完整样本流程，仅减少bootstrap次数并跳过广泛稳健性，输出明确标为运行验证。


## Configuration and input provenance

In [ ]:
import os,sys,json,time,hashlib,socket,platform,re,warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy import stats,linalg
import sklearn,statsmodels
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss,brier_score_loss,roc_auc_score,average_precision_score
from sklearn.calibration import calibration_curve
from statsmodels.stats.multitest import multipletests
from IPython.display import display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
START=time.perf_counter()
ROOT=Path(os.environ.get('WALLET_MODEL_ROOT','.')).resolve()
MODE=os.environ.get('WALLET_MODEL_MODE','production')
OUT=Path(os.environ.get('WALLET_MODEL_OUTPUT',str(ROOT/'results/modeling'/MODE)))
OUT.mkdir(parents=True,exist_ok=True)
CONFIG={'mode':MODE,'seed':20261019,'validation_start':'2026-10-03T00:00:00Z',
 'test_start':'2026-10-04T00:00:00Z','profit_epsilon':1e-6,'bootstrap_replicates':20 if MODE=='smoke' else 300,
 'logistic_C':1.0,'logistic_max_iter':500,'nonlinear_iterations':100,'classification_threshold':0.5,
 'label_definition':'conditional gross profit under unverified payout/ledger assumptions',
 'resolution_eligibility':'recorded resolution proxy only; actual result-publication time unknown'}
def find_input(name):
    for p in [ROOT/'data'/name,ROOT/'server_run/round03'/name,ROOT/'results/full_week'/name]:
        if p.exists():return p
    raise FileNotFoundError(name)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
input_path=find_input('wallet_market_candidates_full_week.parquet')
metadata_path=find_input('weekly_market_metadata_complete.csv')
d=pd.read_parquet(input_path)
metadata=pd.read_csv(metadata_path)
assert len(d)==96184 and d.conditional_profit_label.notna().sum()==89736
assert d.confirmed_net_profit_label.isna().all()
assert not d[['wallet','market']].duplicated().any()
SOURCE_FEATURES=[c for c in d if c.startswith('feature_')]
assert len(SOURCE_FEATURES)==49
for c in ['prediction_time','feature_week_history_first_time','feature_week_history_latest_time']:
    d[c]=pd.to_datetime(d[c],utc=True)
metadata['resolved_at']=pd.to_datetime(metadata.resolved_at,utc=True,format='mixed',errors='coerce')
provenance={'config':CONFIG,'inputs':{str(input_path):sha(input_path),str(metadata_path):sha(metadata_path)},
 'versions':{'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,
 'sklearn':sklearn.__version__,'statsmodels':statsmodels.__version__},'hostname':socket.gethostname(),
 'scheduler_job_id':os.environ.get('JOB_ID'),'platform':platform.platform()}
print('运行模式:',MODE,'；输入组合:',len(d),'；条件标签:',d.conditional_profit_label.notna().sum(),
      '；真实净利润标签:',d.confirmed_net_profit_label.notna().sum())


## 时间、事件与标签成熟度

先用所有输入的事件键和入场时间分组，不看Y：有Oct4以后入场的事件组分配测试，只评估晚入场行、剔除其更早成员。剩余组中有Oct3入场的组分配验证，只评估Oct3且Oct4前记录结算的行。训练来自其余事件、Oct3前入场且记录结算。最终预测重拟合使用Oct4前已经记录结算的非测试事件行。这个后时段优先分配保留了长期市场的晚入场者，又保证市场/事件不跨训练和测试；实际丢失与纳入行全部导出。

体育按日期比赛前缀、天气按城市日期、crypto up/down跨资产按UTC日保守合并，其他阈值市场去掉strike；未识别市场退回市场键。规则不是权威事件ID，剩余跨组相关仍可能存在。训练/测试同钱包是允许的，但只用预测前历史；另报未见钱包切片。


In [ ]:
def event_proxy(slug):
    weather=re.match(r'((?:highest|lowest)-temperature-in-.+?-on-[a-z]+-\d+-\d{4})(?:-|$)',slug)
    if weather:return weather[1],'weather_city_date_proxy'
    dated=re.match(r'(.+?-\d{4}-\d{2}-\d{2})(?:-|$)',slug)
    if dated:return dated[1],'dated_match_prefix_proxy'
    crypto=re.match(r'([a-z]+)-updown-(?:5m|15m|1h|4h)-(\d+)$',slug)
    if crypto:return 'crypto-day-'+str(int(crypto[2])//86400),'all_asset_crypto_UTC_day_proxy'
    threshold=re.match(r'(.+?)-above-\d+(-on-.+)$',slug)
    if threshold:return threshold[1]+threshold[2],'asset_time_threshold_proxy'
    return slug,'unmapped_market_as_proxy'
events=pd.DataFrame([{'market':m,'event_group':event_proxy(m)[0],'event_mapping_rule':event_proxy(m)[1]} for m in d.market.unique()])
assert not events.market.duplicated().any()
d=d.merge(events,on='market',validate='many_to_one').merge(metadata[['market','resolved_at']],on='market',validate='many_to_one')
T0=pd.Timestamp(CONFIG['validation_start']);T1=pd.Timestamp(CONFIG['test_start'])
# Assignment depends on event IDs and all observed entry times, never on Y or profitability eligibility.
test_events=set(d.loc[d.prediction_time.ge(T1),'event_group'])
validation_events=set(d.loc[d.prediction_time.ge(T0)&d.prediction_time.lt(T1)&~d.event_group.isin(test_events),'event_group'])
d['assigned_partition']='development_train'
d.loc[d.event_group.isin(validation_events),'assigned_partition']='validation'
d.loc[d.event_group.isin(test_events),'assigned_partition']='test'
d['eligible_conditional']=d.conditional_profit_label.notna()&d.resolved_at.notna()&d.prediction_time.lt(d.resolved_at)
d['split']='excluded'
d.loc[d.eligible_conditional&d.assigned_partition.eq('development_train')&d.prediction_time.lt(T0)&d.resolved_at.lt(T0),'split']='train'
d.loc[d.eligible_conditional&d.assigned_partition.eq('validation')&d.prediction_time.ge(T0)&d.prediction_time.lt(T1)&d.resolved_at.lt(T1),'split']='validation'
d.loc[d.eligible_conditional&d.assigned_partition.eq('test')&d.prediction_time.ge(T1),'split']='test'
d['final_refit']=d.eligible_conditional&~d.event_group.isin(test_events)&d.prediction_time.lt(T1)&d.resolved_at.lt(T1)
d['exclusion_reason']='retained'
d.loc[~d.conditional_profit_label.notna(),'exclusion_reason']='conditional_label_unknown_negative_inventory'
d.loc[d.conditional_profit_label.notna()&d.resolved_at.isna(),'exclusion_reason']='recorded_resolution_missing'
d.loc[d.conditional_profit_label.notna()&d.resolved_at.notna()&d.prediction_time.ge(d.resolved_at),'exclusion_reason']='entry_at_or_after_recorded_resolution'
d.loc[d.eligible_conditional&d.split.eq('excluded'),'exclusion_reason']='preboundary_member_of_heldout_event_or_training_label_not_mature'
train=d[d.split.eq('train')].copy();val=d[d.split.eq('validation')].copy();test=d[d.split.eq('test')].copy();refit=d[d.final_refit].copy()
assert len(train)>0 and len(val)>0 and len(test)>0 and len(refit)>0
for left,right in [(train,val),(train,test),(val,test),(refit,test)]:
    assert not set(left.event_group)&set(right.event_group)
    assert not set(left.market)&set(right.market)
assert train.prediction_time.max()<T0 and train.resolved_at.max()<T0
assert val.prediction_time.min()>=T0 and val.prediction_time.max()<T1 and val.resolved_at.max()<T1
assert test.prediction_time.min()>=T1 and refit.prediction_time.max()<T1 and refit.resolved_at.max()<T1
assert test.prediction_time.min()>refit.prediction_time.max()
for frame in [train,val,test,refit]:
    assert frame.conditional_profit_label.isin([0,1]).all()
    if frame.feature_week_history_latest_time.notna().any():
        assert (frame.loc[frame.feature_week_history_latest_time.notna(),'feature_week_history_latest_time']<frame.loc[frame.feature_week_history_latest_time.notna(),'prediction_time']).all()
cohort_summary=[]
for name,z in [('all_input',d),('train',train),('validation',val),('test',test),('final_refit',refit)]:
    cohort_summary.append({'cohort':name,'rows':len(z),'wallets':z.wallet.nunique(),'markets':z.market.nunique(),
      'event_proxies':z.event_group.nunique(),'labelled':int(z.conditional_profit_label.notna().sum()),
      'positive_rate_conditional':z.conditional_profit_label.mean(),
      'first_entry':z.prediction_time.min(),'last_entry':z.prediction_time.max(),'last_recorded_resolution':z.resolved_at.max()})
cohort_summary=pd.DataFrame(cohort_summary);display(cohort_summary);cohort_summary.to_csv(OUT/'cohort_summary.csv',index=False)
event_audit=d.groupby('event_group').agg(markets=('market','nunique'),rows=('wallet','size'),
 first_entry=('prediction_time','min'),last_entry=('prediction_time','max'),last_recorded_resolution=('resolved_at','max'),
 assigned_partition=('assigned_partition','first'),event_rule=('event_mapping_rule','first'))
events.to_csv(OUT/'event_mapping.csv',index=False);event_audit.to_csv(OUT/'event_split_audit.csv')
d[['wallet','market','prediction_time','resolved_at','event_group','assigned_partition','eligible_conditional','split','final_refit','exclusion_reason']].to_csv(OUT/'row_split_audit.csv',index=False)
d.groupby('exclusion_reason').size().rename('rows').to_csv(OUT/'exclusion_counts.csv')
print('测试集中与最终训练集相同钱包的比例:',test.wallet.isin(refit.wallet).mean())
print('所有事件组为透明规则代理，不是经核验的真实事件ID；真实结果公开时间仍未知。')



## 固定的特征与变换

当前投入、价格三次多项式、首次批次填单数、严格市场背景、观察年龄/时钟/窗口暴露作为基线；加上钱包周历史活跃、市场广度、买入规模、HHI、跨度及近期程度。价格必须有非线性控制，避免把接近1价格的会计毛利误解释为钱包技能。历史fill与market数高度相关，分别报告条件系数及宽区间，并以时间戳替代活跃度作稳健性。

只在对应训练集拟合中位数、均值和尺度；缺失标记预先定义。测试标签、事后库存、PNL、最终交易额和审计标记不进入特征。极罕见方向集中度用于预测并另作稀疏探索性推断，不进入主关联检验。


In [ ]:
# Tiny chronological event-purge fixture, independent of profitability labels.
split_fixture=pd.DataFrame({'event':['late','late','validation','validation','train','train'],
                            'time':[1,5,2,3,1,2]})
fixture_test=set(split_fixture.loc[split_fixture.time.ge(5),'event'])
fixture_val=set(split_fixture.loc[split_fixture.time.ge(3)&split_fixture.time.lt(5)&~split_fixture.event.isin(fixture_test),'event'])
assert fixture_test=={'late'} and fixture_val=={'validation'}
assert len(split_fixture[split_fixture.event.isin(fixture_test)&split_fixture.time.lt(5)])==1
assert not set(split_fixture.loc[~split_fixture.event.isin(fixture_test|fixture_val),'event'])&(fixture_test|fixture_val)
# Raw field names are the complete allowlist; constructed features never inspect final labels/audits.
BASE_RAW=['feature_entry_log_buy_dollars','feature_entry_buy_vwap','feature_entry_direction_concentration',
 'feature_entry_fill_rows','feature_strict_prior_market_log_dollars','feature_strict_prior_market_log_wallets',
 'feature_observed_market_age_hours','feature_entry_utc_hour']
HISTORY_RAW=['feature_week_history_log_fill_rows','feature_week_history_log_markets',
 'feature_week_history_log_buy_dollars','feature_week_history_buy_market_hhi',
 'feature_week_history_observed_span_hours','feature_week_history_log_recency_hours']
assert all(c.startswith('feature_') and c in SOURCE_FEATURES for c in BASE_RAW+HISTORY_RAW)
def raw_design(frame,history=True,association=False):
    X=pd.DataFrame(index=frame.index)
    X['entry_log_buy']=frame.feature_entry_log_buy_dollars
    p=frame.feature_entry_buy_vwap
    X['entry_price']=p;X['entry_price_squared']=p**2;X['entry_price_cubed']=p**3
    if not association:X['entry_concentration']=frame.feature_entry_direction_concentration
    X['entry_log_fill_rows']=np.log1p(frame.feature_entry_fill_rows)
    X['market_log_dollars']=frame.feature_strict_prior_market_log_dollars
    X['market_log_wallets']=frame.feature_strict_prior_market_log_wallets
    X['market_log_observed_age']=np.log1p(frame.feature_observed_market_age_hours.clip(lower=0))
    hour=frame.feature_entry_utc_hour
    X['clock_sin']=np.sin(2*np.pi*hour/24);X['clock_cos']=np.cos(2*np.pi*hour/24)
    X['window_exposure_days']=(frame.prediction_time-pd.Timestamp('2026-09-29',tz='UTC')).dt.total_seconds()/86400
    if history:
        X['history_log_fills']=frame.feature_week_history_log_fill_rows
        X['history_log_markets']=frame.feature_week_history_log_markets
        X['history_log_buy']=frame.feature_week_history_log_buy_dollars
        X['history_buy_hhi']=frame.feature_week_history_buy_market_hhi
        X['history_log_span']=np.log1p(frame.feature_week_history_observed_span_hours.clip(lower=0))
        X['history_log_recency']=frame.feature_week_history_log_recency_hours
    for name in ['entry_price','history_buy_hhi','history_log_span','history_log_recency']:
        if name in X:X[name+'__missing']=X[name].isna().astype(float)
    assert not any(re.search(r'label|audit|payout|cashflow|final',c) for c in X.columns)
    return X.astype(float)
class TrainPreprocessor:
    def fit(self,X):
        self.columns=list(X);self.median=X.median().fillna(0.)
        filled=X.fillna(self.median)
        self.mean=filled.mean();self.scale=filled.std(ddof=0).replace(0,1.)
        for c in X:
            if c.endswith('__missing'):self.mean[c]=0.;self.scale[c]=1.
        self.fit_rows=len(X)
        return self
    def transform(self,X):
        assert list(X)==self.columns
        return ((X.fillna(self.median)-self.mean)/self.scale).to_numpy(dtype=float)
# A test-only outlier cannot change fitted medians/means.
fixture=pd.DataFrame({'x':[1.,np.nan,3.]})
pp=TrainPreprocessor().fit(fixture);before=pp.median.copy();pp.transform(pd.DataFrame({'x':[1e12]}))
pd.testing.assert_series_equal(before,pp.median)


## 条件关联方法与验证

主分析是市场固定效应LPM，按市场去均值后估计，不创建巨大稠密虚拟变量。`beta`以训练标准差为连续变量单位，缺失标记为0→1，报告概率百分点。其目的为关联，不用于最终概率预测；LPM可超出[0,1]。同一标签市场仍可贡献斜率分母，单独报告数量；这与条件logit不同。

协方差按钱包和启发式事件两维聚类：V=V(wallet)+V(event)−V(intersection)。各项校正为G/(G−1)×(n−1)/(n−K−M)，K为保留斜率数、M为吸收的市场自由度；参照t自由度=min(钱包簇,事件簇)−1。小fixture与完整dummy OLS的statsmodels结果对齐。报告簇不平衡和有效簇数；没有把代理事件当真实独立性证明。整体协方差可能非正定，不作PSD修补；非正定/奇异概念子块的联合检验不可用。

主Holm检验族按预先规定的概念，价格三项和时钟两项联合检验，缺失标记成组；活跃/市场广度分别单项，相关性通过条件估计/区间暴露。不可估计的概念按p=1保留族大小、最终展示为缺失。另一个明确标为探索的系数族提供单项Holm/BH。多重检验不能解决标签错误或选择偏差。


In [ ]:
CONCEPTS={
 'entry_buy_scale':['entry_log_buy'],'entry_price_nonlinear':['entry_price','entry_price_squared','entry_price_cubed'],
 'entry_fill_rows':['entry_log_fill_rows'],'prior_market_volume':['market_log_dollars'],
 'prior_market_participants':['market_log_wallets'],'observed_market_age':['market_log_observed_age'],
 'utc_clock':['clock_sin','clock_cos'],'observation_window_exposure':['window_exposure_days'],
 'history_activity':['history_log_fills'],'history_breadth':['history_log_markets'],'history_buy_scale':['history_log_buy'],
 'history_buy_concentration':['history_buy_hhi'],'history_observed_span':['history_log_span'],
 'history_recency':['history_log_recency'],
 'missingness':['entry_price__missing','history_buy_hhi__missing','history_log_span__missing','history_log_recency__missing']}
def within(A,group):
    return A-pd.DataFrame(A).groupby(np.asarray(group),sort=False).transform('mean').to_numpy()
def cluster_meat(scores,groups):
    codes,_=pd.factorize(pd.Index(groups,tupleize_cols=False),sort=False)
    grouped=np.zeros((codes.max()+1,scores.shape[1]));np.add.at(grouped,codes,scores)
    return grouped.T@grouped,len(grouped)
def covariance_2way(X,resid,wallet,event,absorbed_groups):
    n,k=X.shape;bread=np.linalg.inv(X.T@X);scores=X*resid[:,None]
    terms=[];counts=[]
    for groups in [wallet,event,list(zip(wallet,event))]:
        meat,G=cluster_meat(scores,groups)
        if G<2 or n-k-absorbed_groups<=0:raise ValueError('Insufficient clusters or residual degrees of freedom')
        correction=G/(G-1)*(n-1)/(n-k-absorbed_groups)
        terms.append(correction*bread@meat@bread);counts.append(G)
    C=terms[0]+terms[1]-terms[2]
    return (C+C.T)/2,counts

def fit_association(frame,y,name,include_rare_direction=False):
    raw=raw_design(frame,history=True,association=not include_rare_direction)
    pre=TrainPreprocessor().fit(raw);matrix=pre.transform(raw)
    market=frame.market.to_numpy();G=frame.market.nunique();Xw=within(matrix,market)
    yw=within(np.asarray(y,dtype=float)[:,None],market).ravel()
    # Prespecified order retained; exact zero/alias columns dropped by rank, never p-value.
    keep=[];drops=[];rank=0
    for j,c in enumerate(raw.columns):
        trial=Xw[:,keep+[j]];newrank=np.linalg.matrix_rank(trial,tol=1e-8)
        if np.linalg.norm(Xw[:,j])<=1e-8 or newrank<=rank:drops.append({'column':c,'reason':'zero within-market variation or exact linear alias'})
        else:keep.append(j);rank=newrank
    names=[raw.columns[j] for j in keep];X=Xw[:,keep]
    beta=np.linalg.lstsq(X,yw,rcond=None)[0];resid=yw-X@beta
    C,clusters=covariance_2way(X,resid,frame.wallet.to_numpy(),frame.event_group.to_numpy(),G)
    dfref=min(clusters[:2])-1
    eigen=np.linalg.eigvalsh(C);valid=np.diag(C)>0
    se=np.where(valid,np.sqrt(np.maximum(np.diag(C),0)),np.nan)
    t=beta/se;p=2*stats.t.sf(np.abs(t),dfref);critical=stats.t.ppf(.975,dfref)
    coef=pd.DataFrame({'term':names,'coefficient':beta,'se_2way':se,'t':t,'df_reference':dfref,'p_exploratory':p,
        'ci95_low':beta-critical*se,'ci95_high':beta+critical*se,
        'ci95_low_percentage_points':100*(beta-critical*se),'ci95_high_percentage_points':100*(beta+critical*se),'effect_percentage_points':100*beta,'units':['0 to 1 missingness indicator' if c.endswith('__missing') else 'one development-training SD' for c in names]})
    ok=np.isfinite(coef.p_exploratory)
    coef['p_holm_exploratory_family']=np.nan;coef['q_bh_exploratory_family']=np.nan
    if ok.any():
        coef.loc[ok,'p_holm_exploratory_family']=multipletests(coef.loc[ok,'p_exploratory'],method='holm')[1]
        coef.loc[ok,'q_bh_exploratory_family']=multipletests(coef.loc[ok,'p_exploratory'],method='fdr_bh')[1]
    tests=[]
    for concept,terms in CONCEPTS.items():
        retained=[c for c in terms if c in names]
        rec={'concept':concept,'specified_terms':';'.join(terms),'retained_terms':';'.join(retained),'df_num':len(retained),'p_raw':np.nan,'status':'unavailable_all_terms_aliased'}
        if retained:
            idx=[names.index(c) for c in retained];sub=C[np.ix_(idx,idx)];v=beta[idx]
            eig=np.linalg.eigvalsh(sub)
            if eig.min()>1e-12:
                F=float(v@np.linalg.solve(sub,v)/len(idx))
                rec.update({'F':F,'p_raw':stats.f.sf(F,len(idx),dfref),'status':'estimable' if len(retained)==len(terms) else 'estimable_reduced_due_alias'})
            else:rec['status']='unavailable_nonpositive_or_singular_cluster_covariance'
        tests.append(rec)
    tests=pd.DataFrame(tests);ok=tests.p_raw.notna()
    # Frozen family includes unavailable tests conservatively as p=1 for Holm's family size.
    tests['p_holm_primary_family']=multipletests(tests.p_raw.fillna(1),method='holm')[1]
    tests.loc[~ok,'p_holm_primary_family']=np.nan
    tests['q_bh_primary_family']=multipletests(tests.p_raw.fillna(1),method='fdr_bh')[1]
    tests.loc[~ok,'q_bh_primary_family']=np.nan
    event_sizes=frame.event_group.value_counts(normalize=True)
    diagnostics={'analysis':name,'rows':len(frame),'markets':G,'wallet_clusters':clusters[0],
      'event_proxy_clusters':clusters[1],'intersection_clusters':clusters[2],'df_reference':dfref,
      'absorbed_fe_degrees':G,'slope_rank':len(names),'residual_df':len(frame)-G-len(names),
      'within_condition_number':float(np.linalg.cond(X)), 'covariance_min_eigenvalue':float(eigen.min()),
      'covariance_negative_eigenvalues':int((eigen < -1e-12).sum()),
      'effective_event_clusters_inverse_hhi':float(1/(event_sizes**2).sum()),
      'largest_event_share':float(event_sizes.max()),'single_outcome_markets':int(frame.assign(y=np.asarray(y)).groupby('market').y.nunique().eq(1).sum()),
      'within_residual_sum':float(resid.sum()),'lpm_conditional_fitted_outside_01':int(((np.asarray(y)-resid<0)|(np.asarray(y)-resid>1)).sum())}
    coef.to_csv(OUT/(name+'_coefficients.csv'),index=False);tests.to_csv(OUT/(name+'_concept_tests.csv'),index=False)
    pd.DataFrame(drops,columns=['column','reason']).to_csv(OUT/(name+'_dropped_terms.csv'),index=False)
    Path(OUT/(name+'_diagnostics.json')).write_text(json.dumps(diagnostics,indent=2,default=str))
    return coef,tests,diagnostics,pre
# Small covariance fixture: absorbed slopes match full dummy OLS, including absorbed FE df.
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups
rng=np.random.default_rng(12);n=160;g=np.repeat(np.arange(8),20);wa=np.tile(np.arange(20),8);ev=g//2
xf=rng.normal(size=(n,2));yf=xf@np.array([.3,-.2])+g*.1+rng.normal(size=n)
xw=within(xf,g);yw=within(yf[:,None],g).ravel();bf=np.linalg.lstsq(xw,yw,rcond=None)[0]
cf,_=covariance_2way(xw,yw-xw@bf,wa,ev,len(np.unique(g)))
dummies=pd.get_dummies(g,dtype=float).to_numpy();fixture_fit=sm.OLS(yf,np.column_stack([xf,dummies])).fit()
refcov=cov_cluster_2groups(fixture_fit,wa,ev,use_correction=True)[0][:2,:2]
assert np.allclose(bf,fixture_fit.params[:2],atol=1e-10)
assert np.allclose(cf,refcov,atol=1e-10)
assert np.allclose(within(xf,g).reshape(8,20,2).mean(axis=1),0,atol=1e-12)
print('Within-market and two-way covariance/dummy-OLS fixtures passed.')


## 49列登记与可识别性

逐列提供定义、单位、时间口径、训练缺失/取值支持、选用或替代原因，以及对应概念。没有单独检验的列不称为“不显著”。原值/log值、历史绝对时间/跨度、方向HHI/集中度等共享含义。0/1结果方向的命名也没有统一跨市场语义。


In [ ]:
FIELD_TO_CONCEPT={
 'feature_entry_buy_dollars':'entry_buy_scale','feature_entry_log_buy_dollars':'entry_buy_scale',
 'feature_entry_buy_vwap':'entry_price_nonlinear','feature_entry_fill_rows':'entry_fill_rows',
 'feature_entry_buy_shares':'entry_scale_price_alias','feature_entry_sell_dollars':'rare_entry_sell',
 'feature_entry_has_sell':'rare_entry_sell','feature_entry_no_buy':'constant_in_conditional_cohort',
 'feature_entry_outcome0_buy_dollar_share':'outcome_index_orientation',
 'feature_entry_direction_concentration':'rare_direction_pattern','feature_entry_both_outcomes_bought':'rare_direction_pattern',
 'feature_entry_direction_hhi':'rare_direction_pattern',
 'feature_strict_prior_market_dollars':'prior_market_volume','feature_strict_prior_market_log_dollars':'prior_market_volume',
 'feature_strict_prior_market_wallets':'prior_market_participants','feature_strict_prior_market_log_wallets':'prior_market_participants',
 'feature_observed_market_age_hours':'observed_market_age','feature_entry_utc_hour':'utc_clock',
 'feature_entry_utc_weekday':'observation_window_exposure','feature_entry_utc_date':'observation_window_exposure',
 'feature_entry_total_dollars':'entry_scale_sell_alias','feature_entry_sell_dollar_fraction':'rare_entry_sell',
 'feature_week_history_fill_rows':'history_activity','feature_week_history_log_fill_rows':'history_activity',
 'feature_week_history_distinct_timestamps':'history_activity_alternative','feature_week_history_log_distinct_timestamps':'history_activity_alternative',
 'feature_week_history_markets':'history_breadth','feature_week_history_log_markets':'history_breadth',
 'feature_week_history_buy_dollars':'history_buy_scale','feature_week_history_log_buy_dollars':'history_buy_scale',
 'feature_week_history_buy_market_hhi':'history_buy_concentration',
 'feature_week_history_first_time':'history_observed_span','feature_week_history_observed_span_hours':'history_observed_span',
 'feature_week_history_latest_time':'history_recency','feature_week_history_recency_hours':'history_recency',
 'feature_week_history_log_recency_hours':'history_recency','feature_week_history_no_observed_rows':'missingness',
 'feature_week_history_no_buy':'missingness','feature_week_history_prior_conditional_profit_history_available':'unavailable_historical_outcomes'}
for c in SOURCE_FEATURES:
    if c.startswith('feature_entry_outcome') and c not in FIELD_TO_CONCEPT:FIELD_TO_CONCEPT[c]='outcome_specific_encoding'
assert set(FIELD_TO_CONCEPT)==set(SOURCE_FEATURES)
registry=[]
for c in SOURCE_FEATURES:
    concept=FIELD_TO_CONCEPT[c];selected=c in BASE_RAW+HISTORY_RAW
    if selected:
        status='selected_predictor';reason='Prespecified compact concept; transformed using training data only'
        if c=='feature_entry_direction_concentration':status='prediction_only_rare_support';reason='Only rare two-sided entries differ; omitted primary inference; sparse-pattern sensitivity'
    else:
        status='alternative_or_alias';reason='Represented by log/aggregate/duration encoding of same concept; no separate independent p-value'
        if concept in ['rare_entry_sell','rare_direction_pattern']:status='rare_descriptive';reason='Very limited effective support; cannot claim stable broad independent association'
        elif concept.startswith('outcome_'):status='semantic_alternative';reason='Catalog outcome index not common bullish/Yes direction across markets; aggregate price/scale used'
        elif 'zero_start' in c:status='unverified_entry_position_proxy';reason='Zero-start observed net shares are not verified holdings; redundant with first-batch trade scale'
        elif concept=='constant_in_conditional_cohort':status='constant';reason='No-buy is constant in eligible conditional cohort'
        elif concept=='unavailable_historical_outcomes':status='unavailable';reason='Constant zero: past trusted profit/result-publication data unavailable'
        elif c.endswith('first_time') or c.endswith('latest_time'):reason='Absolute time encoded as strict-prior duration/recency, not raw timestamp'
    registry.append({'source_feature':c,'concept':concept,'status':status,'reason':reason,
      'dtype':str(d[c].dtype),'missing_all':d[c].isna().mean(),'missing_train':train[c].isna().mean(),
      'unique_train':train[c].nunique(dropna=True),'nonzero_train':int(train[c].fillna(0).ne(0).sum()) if pd.api.types.is_numeric_dtype(train[c]) else np.nan,
      'significance_reference':'Primary concept table; individual terms exploratory' if concept in CONCEPTS else 'No standalone primary test',
      'prediction_reference':'Prespecified concept-drop table if modeled; aliases share concept',
      'time_availability':'First observed batch or strict-prior supplied-week history/context; historical source coverage unverified'})
registry=pd.DataFrame(registry)
# Attach original formula/units if the authoritative dictionary is staged.
try:
    dictionary_path=find_input('wallet_market_candidates_feature_dictionary.csv')
    old_dictionary=pd.read_csv(dictionary_path)
    registry=registry.merge(old_dictionary[['column','units','definition']],left_on='source_feature',right_on='column',how='left',validate='one_to_one').drop(columns='column')
    provenance['inputs'][str(dictionary_path)]=sha(dictionary_path)
except FileNotFoundError:
    registry['units']='See source feature definition';registry['definition']=registry.source_feature.str.replace('_',' ')
assert len(registry)==49 and registry.source_feature.nunique()==49
registry.to_csv(OUT/'all_49_feature_registry.csv',index=False)
display(registry[['source_feature','concept','status','missing_train','unique_train']])
# Descriptive comparison of accounting-ineligible and eligible entry cohorts, not predictors.
support=[]
for c in ['feature_entry_has_sell','feature_entry_no_buy','feature_entry_both_outcomes_bought','feature_entry_direction_concentration','feature_entry_buy_vwap']:
    for label,z in [('conditional_label_present',d[d.conditional_profit_label.notna()]),('conditional_label_unknown',d[d.conditional_profit_label.isna()])]:
        support.append({'feature':c,'cohort':label,'rows':len(z),'nonmissing':z[c].notna().sum(),'mean':z[c].mean(),'unique':z[c].nunique()})
pd.DataFrame(support).to_csv(OUT/'feature_support_and_selection.csv',index=False)


## 主条件关联结果

只用初始开发训练集做关联，测试集不参与系数/不确定性/缺失填补估计。正负系数表示控制其他选定项和市场后，条件标签概率的关联方向。宽区间、高条件数、极少有效事件或非正定协方差均限制解释；p<.05不是钱包能力、因果关系或精确真实净利润证据。


In [ ]:
primary_coef,primary_tests,primary_diag,primary_pre=fit_association(train,train.conditional_profit_label.astype(int),'primary_development_fe_lpm')
print('主分析是市场固定效应线性概率模型；系数不是logit优势比，也不是因果效应。')
print('聚类簇数与有效簇数可能差异很大；共享crypto日可造成极端不平衡。t/F和bootstrap区间是近似，不因行数大就可靠。')
display(pd.DataFrame([primary_diag]));display(primary_tests)
display(primary_coef[['term','effect_percentage_points','ci95_low_percentage_points','ci95_high_percentage_points','p_exploratory','p_holm_exploratory_family']])
if primary_diag['covariance_negative_eigenvalues']:
    print('注意：两维聚类协方差整体非正定。没有做PSD修补；只有对应协方差子块正定的概念检验才报告。')


## 概率损失与事件重采样

主损失为log loss/Brier，辅以ROC-AUC、PR-AUC和校准。正的Δ损失=删去特征后的损失−完整模型损失，表示完整模型更好。固定模型的预测按事件代理成组有放回抽样，保留组内行；区间只描述本测试事件的条件波动，不包括重训练不确定性，也不证明未来总体泛化。没有逐行独立bootstrap或把预测贡献当p值。


In [ ]:
MODEL_CHECKS=[]
def fit_logistic(X,y,label):
    model=LogisticRegression(C=CONFIG['logistic_C'],max_iter=CONFIG['logistic_max_iter'],solver='lbfgs',random_state=CONFIG['seed'])
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always');model.fit(X,np.asarray(y,dtype=int))
    MODEL_CHECKS.append({'model':label,'iterations':int(model.n_iter_.max()),'converged':bool(model.n_iter_.max()<CONFIG['logistic_max_iter']),
                         'warnings':';'.join(str(w.message) for w in ws),'rows':len(y),'features':X.shape[1]})
    return model

def probability_metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.clip(np.asarray(p),1e-12,1-1e-12)
    binary=(p>=CONFIG['classification_threshold']).astype(int)
    return {'rows':len(y),'positive_rate':float(y.mean()),'log_loss':log_loss(y,p,labels=[0,1]),
       'brier':brier_score_loss(y,p),'roc_auc':roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
       'pr_auc_average_precision':average_precision_score(y,p) if y.sum()>0 else np.nan,
       'mean_prediction':float(p.mean()),'calibration_mean_error':float(p.mean()-y.mean()),
       'accuracy_threshold_0_5':float((binary==y).mean())}

def paired_event_delta(y,reference,alternative,events,seed=20261019):
    # Positive alternative-minus-reference loss means the reference model helped.
    y=np.asarray(y,dtype=float);r=np.clip(reference,1e-12,1-1e-12);a=np.clip(alternative,1e-12,1-1e-12)
    frame=pd.DataFrame({'event':np.asarray(events),'n':1,
      'log_loss_delta':-(y*np.log(a)+(1-y)*np.log(1-a))+(y*np.log(r)+(1-y)*np.log(1-r)),
      'brier_delta':(y-a)**2-(y-r)**2})
    grouped=frame.groupby('event').sum();G=len(grouped);rng=np.random.default_rng(seed)
    draws=rng.integers(0,G,size=(CONFIG['bootstrap_replicates'],G))
    answer={'event_proxy_groups':G,'bootstrap_replicates':CONFIG['bootstrap_replicates'],
      'effective_event_proxy_groups_inverse_hhi':float(1/((grouped.n/grouped.n.sum())**2).sum()),
      'largest_event_row_share':float(grouped.n.max()/grouped.n.sum())}
    for c in ['log_loss_delta','brier_delta']:
        boot=grouped[c].to_numpy()[draws].sum(axis=1)/grouped.n.to_numpy()[draws].sum(axis=1)
        answer[c]=float(frame[c].mean());answer[c+'_ci_low']=float(np.quantile(boot,.025));answer[c+'_ci_high']=float(np.quantile(boot,.975))
    return answer
assert paired_event_delta([0,1],[.2,.8],[.4,.6],['a','b'])==paired_event_delta([0,1],[.2,.8],[.4,.6],['a','b'])
assert paired_event_delta([0,1],[.2,.8],[.4,.6],['a','b'])['log_loss_delta']>0


## 实际预测模型

固定模型：训练流行率常数、当前成交/背景logistic、加入钱包历史logistic，以及固定复杂度Histogram Gradient Boosting。正则logistic用于预测，不附无惩罚回归p值；与市场FE关联模型的统计目标不同。超参数预先固定，验证集只作诊断；未根据测试集挑模型/阈值。测试表报告所有预先规定模型，而非只展示最优者。


In [ ]:
validation_metrics=[];test_metrics=[];test_predictions={};validation_predictions={};fitted_objects={}
for use_history,label in [(False,'current_context_logistic'),(True,'current_context_plus_history_logistic')]:
    pre=TrainPreprocessor().fit(raw_design(train,history=use_history))
    model=fit_logistic(pre.transform(raw_design(train,history=use_history)),train.conditional_profit_label,label+'_validation')
    pv=model.predict_proba(pre.transform(raw_design(val,history=use_history)))[:,1]
    validation_predictions[label]=pv;validation_metrics.append({'model':label,**probability_metrics(val.conditional_profit_label,pv)})
    final_pre=TrainPreprocessor().fit(raw_design(refit,history=use_history))
    final_model=fit_logistic(final_pre.transform(raw_design(refit,history=use_history)),refit.conditional_profit_label,label+'_final')
    pt=final_model.predict_proba(final_pre.transform(raw_design(test,history=use_history)))[:,1]
    test_predictions[label]=pt;test_metrics.append({'model':label,**probability_metrics(test.conditional_profit_label,pt)})
    fitted_objects[label]=(final_pre,final_model)
# Fixed train prevalence baseline; no final-test-label calibration.
validation_metrics.append({'model':'train_prevalence',**probability_metrics(val.conditional_profit_label,np.repeat(train.conditional_profit_label.mean(),len(val)))})
test_predictions['train_prevalence']=np.repeat(refit.conditional_profit_label.mean(),len(test))
test_metrics.append({'model':'train_prevalence',**probability_metrics(test.conditional_profit_label,test_predictions['train_prevalence'])})
# Nonlinear comparator has a frozen specification and is never selected using final-test performance.
for fit_frame,eval_frame,phase in [(train,val,'validation'),(refit,test,'test')]:
    hp=TrainPreprocessor().fit(raw_design(fit_frame,history=True))
    nonlinear=HistGradientBoostingClassifier(max_iter=CONFIG['nonlinear_iterations'],max_leaf_nodes=15,
        learning_rate=.07,l2_regularization=1.,min_samples_leaf=50,early_stopping=False,random_state=CONFIG['seed'])
    nonlinear.fit(hp.transform(raw_design(fit_frame,history=True)),fit_frame.conditional_profit_label.astype(int))
    pred=nonlinear.predict_proba(hp.transform(raw_design(eval_frame,history=True)))[:,1]
    metrics={'model':'fixed_hist_gradient_boosting',**probability_metrics(eval_frame.conditional_profit_label,pred)}
    if phase=='validation':validation_metrics.append(metrics)
    else:test_metrics.append(metrics);test_predictions['fixed_hist_gradient_boosting']=pred
    MODEL_CHECKS.append({'model':'fixed_hist_gradient_boosting_'+phase,'iterations':nonlinear.n_iter_,'converged':'fixed_iteration_budget','rows':len(fit_frame),'features':len(hp.columns)})
validation_metrics=pd.DataFrame(validation_metrics);test_metrics=pd.DataFrame(test_metrics)
validation_metrics.to_csv(OUT/'validation_metrics.csv',index=False);test_metrics.to_csv(OUT/'heldout_test_metrics.csv',index=False)
print('验证集仅作固定模型的诊断；不调参。最终测试集不选择模型、变量或阈值。')
display(validation_metrics);display(test_metrics)
prediction_export=test[['wallet','market','prediction_time','event_group','conditional_profit_label']].copy()
for name,pred in test_predictions.items():prediction_export[name]=pred
prediction_export['wallet_observed_in_final_refit']=test.wallet.isin(refit.wallet)
prediction_export.to_csv(OUT/'heldout_test_predictions.csv',index=False)
history_delta=paired_event_delta(test.conditional_profit_label,test_predictions['current_context_plus_history_logistic'],
 test_predictions['current_context_logistic'],test.event_group)
pd.DataFrame([{'comparison':'remove_all_history_from_logistic',**history_delta}]).to_csv(OUT/'all_history_paired_test_delta.csv',index=False)
display(pd.DataFrame([history_delta]))
wallet_slices=[]
for known in [False,True]:
    mask=test.wallet.isin(refit.wallet).eq(known).to_numpy()
    if mask.any():
        for name,pred in test_predictions.items():wallet_slices.append({'wallet_seen_in_refit':known,'model':name,**probability_metrics(test.loc[mask,'conditional_profit_label'],pred[mask])})
pd.DataFrame(wallet_slices).to_csv(OUT/'heldout_wallet_overlap_slices.csv',index=False)


## 各概念预测贡献

对每个预先指定概念，固定其余特征、训练样本和正则设置，删除该概念后重拟合，在同一冻结测试集比较损失及事件bootstrap区间。价格/时钟按合法多项概念成组，历史fill/market/金额/HHI/跨度/近期分别删除。高相关特征可替代，因此独立显著与删去增益可以不同。这是探索性测试集诊断，不能再据此挑变量并宣称该测试集仍独立。


In [ ]:
PREDICTIVE_DROPS={
 'entry_buy_scale':['entry_log_buy'],'entry_price_nonlinear':['entry_price','entry_price_squared','entry_price_cubed','entry_price__missing'],
 'entry_fill_rows':['entry_log_fill_rows'],'entry_direction_rare':['entry_concentration'],
 'prior_market_volume':['market_log_dollars'],'prior_market_participants':['market_log_wallets'],
 'observed_market_age':['market_log_observed_age'],'utc_clock':['clock_sin','clock_cos'],
 'window_exposure':['window_exposure_days'],'history_activity_fills':['history_log_fills'],
 'history_breadth_markets':['history_log_markets'],'history_buy_scale':['history_log_buy'],
 'history_buy_concentration':['history_buy_hhi','history_buy_hhi__missing'],
 'history_observed_span':['history_log_span','history_log_span__missing'],
 'history_recency':['history_log_recency','history_log_recency__missing']}
full_pre,full_model=fitted_objects['current_context_plus_history_logistic']
Xfinal=full_pre.transform(raw_design(refit,history=True));Xtest=full_pre.transform(raw_design(test,history=True))
ablations=[]
for concept,terms in PREDICTIVE_DROPS.items():
    retained=[j for j,c in enumerate(full_pre.columns) if c not in terms]
    m=fit_logistic(Xfinal[:,retained],refit.conditional_profit_label,'drop_'+concept)
    pred=m.predict_proba(Xtest[:,retained])[:,1]
    ablations.append({'dropped_concept':concept,'dropped_terms':';'.join(terms),
       **probability_metrics(test.conditional_profit_label,pred),
       **paired_event_delta(test.conditional_profit_label,test_predictions['current_context_plus_history_logistic'],pred,test.event_group)})
ablations=pd.DataFrame(ablations);ablations.to_csv(OUT/'exploratory_concept_drop_test_contributions.csv',index=False)
print('预测贡献表：正的Δ损失表示删去该概念后更差。相关特征可替代彼此，因此这与独立显著性不同。所有比较固定，未据此挑选最终模型。')
display(ablations[['dropped_concept','log_loss_delta','log_loss_delta_ci_low','log_loss_delta_ci_high','brier_delta','brier_delta_ci_low','brier_delta_ci_high']])


## 会计与选择稳健性

反转映射与全字段去重分别重算条件目标，保持原有负库存排除口径；重复受影响行排除、价格[.05,.95]限制是改变研究人群的敏感性，不是“正确样本”的证明。另报告稀疏方向和活跃时间戳替代。每个分析导出样本/事件变化和模型结果，不因显著性挑选场景。缺费用时反转/去重仍无法恢复真实净利润。


In [ ]:
sensitivity_records=[];sensitivity_tests=[]
if MODE!='smoke':
    scenarios=[('reverse_mapping','label_reversed_mapping_gross_pnl',None),
      ('dedup_assumption','label_dedup_assumption_gross_pnl',None),
      ('exclude_duplicate_affected','label_assumed_zero_start_gross_pnl','no_duplicates'),
      ('entry_price_0_05_to_0_95','label_assumed_zero_start_gross_pnl','mid_price')]
    for name,pnl_column,restriction in scenarios:
        def restrict(frame):
            if restriction=='no_duplicates':return frame[frame.audit_only_duplicate_excess.eq(0)].copy()
            if restriction=='mid_price':return frame[frame.feature_entry_buy_vwap.between(.05,.95)].copy()
            return frame.copy()
        st=restrict(train);sr=restrict(refit);se=restrict(test)
        # Identical preexisting accounting mask: each scenario is still conditional, not net truth.
        yt=st[pnl_column].gt(CONFIG['profit_epsilon']).astype(int)
        yr=sr[pnl_column].gt(CONFIG['profit_epsilon']).astype(int)
        ye=se[pnl_column].gt(CONFIG['profit_epsilon']).astype(int)
        if min(len(st),len(sr),len(se))==0 or min(yt.nunique(),yr.nunique())<2:
            sensitivity_records.append({'scenario':name,'status':'unavailable_insufficient_outcome_support'});continue
        co,te,di,_=fit_association(st,yt,'sensitivity_'+name)
        te['scenario']=name;sensitivity_tests.append(te)
        preds={}
        for use_history,label in [(False,'baseline'),(True,'full_history')]:
            pre=TrainPreprocessor().fit(raw_design(sr,history=use_history))
            model=fit_logistic(pre.transform(raw_design(sr,history=use_history)),yr,name+'_'+label)
            pred=model.predict_proba(pre.transform(raw_design(se,history=use_history)))[:,1];preds[label]=pred
            sensitivity_records.append({'scenario':name,'model':label,'status':'fit',
                'training_rows':len(st),'final_refit_rows':len(sr),'test_events':se.event_group.nunique(),
                'test_markets':se.market.nunique(),'target_positive_rate':ye.mean(),**probability_metrics(ye,pred)})
        delta=paired_event_delta(ye,preds['full_history'],preds['baseline'],se.event_group)
        Path(OUT/('sensitivity_'+name+'_history_delta.json')).write_text(json.dumps(delta,indent=2,default=str))
    # Sparse direction estimate is explicitly exploratory, excluded from primary family.
    rare_co,_,rare_diag,_=fit_association(train,train.conditional_profit_label.astype(int),
       'sensitivity_rare_direction',include_rare_direction=True)
    print('方向集中度的探索性估计依赖极少数非单边首次买入，不作广泛独立能力结论。')
    # Alternative activity encoding: timestamps replaces fills, same concept family.
    alternative=train.copy()
    alternative['feature_week_history_log_fill_rows']=alternative.feature_week_history_log_distinct_timestamps
    fit_association(alternative,alternative.conditional_profit_label.astype(int),'sensitivity_activity_timestamps')
else:
    sensitivity_records.append({'scenario':'broad_robustness','status':'smoke_mode_skipped; production required'})
sensitivity_metrics=pd.DataFrame(sensitivity_records);sensitivity_metrics.to_csv(OUT/'sensitivity_prediction_metrics.csv',index=False)
if sensitivity_tests:pd.concat(sensitivity_tests,ignore_index=True).to_csv(OUT/'sensitivity_concept_tests_all.csv',index=False)
display(sensitivity_metrics)


## 图表与解释边界

汇总的主结果均由下方实际计算输出。记录输入哈希、软件版本、完整配置、调度JOB_ID、时间与输出哈希。正式结论必须同时结合选择机制、事件代理、聚类不平衡、共线性和会计条件，不能只按显著与否分类。

方法参考：[statsmodels两维聚类](https://www.statsmodels.org/stable/generated/statsmodels.stats.sandwich_covariance.cov_cluster_2groups.html)、[sklearn避免泄漏](https://scikit-learn.org/stable/common_pitfalls.html)、[sklearn时间/组验证](https://scikit-learn.org/stable/modules/cross_validation.html)。


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].barh(test_metrics.model,test_metrics.log_loss);axes[0].set_xlabel('Held-out log loss (lower is better)')
for name,pred in test_predictions.items():
    if len(np.unique(test.conditional_profit_label))>1:
        frac,mean=calibration_curve(test.conditional_profit_label.astype(int),pred,n_bins=8,strategy='quantile')
        axes[1].plot(mean,frac,'o-',label=name)
axes[1].plot([0,1],[0,1],'k--',linewidth=.8);axes[1].set(xlabel='Mean predicted probability',ylabel='Observed conditional positive fraction')
axes[1].legend(fontsize=7);fig.tight_layout();fig.savefig(OUT/'heldout_performance_calibration.png',dpi=150);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6));z=ablations.iloc[::-1]
ax.plot(z.log_loss_delta,np.arange(len(z)),'o')
ax.hlines(np.arange(len(z)),z.log_loss_delta_ci_low,z.log_loss_delta_ci_high)
ax.axvline(0,color='black',linewidth=.8);ax.set_yticks(np.arange(len(z)),z.dropped_concept)
ax.set_xlabel('Drop-concept minus full-model held-out log loss; event-bootstrap 95% interval')
fig.tight_layout();fig.savefig(OUT/'concept_prediction_contributions.png',dpi=150);plt.close(fig)
from IPython.display import Image
for image_name in ['heldout_performance_calibration.png','concept_prediction_contributions.png']:display(Image(filename=str(OUT/image_name)))
checks=pd.DataFrame(MODEL_CHECKS);checks.to_csv(OUT/'model_convergence_checks.csv',index=False)
print('条件显著性与预测贡献是不同问题。主概念Holm校正后可报告的关联数:',
      int(primary_tests.p_holm_primary_family.lt(.05).sum()),'/',int(primary_tests.p_raw.notna().sum()))
print('全部历史相对当前交易/环境基线的测试Δlogloss（基线−完整）:',history_delta['log_loss_delta'],
      '；事件重采样区间:',history_delta['log_loss_delta_ci_low'],history_delta['log_loss_delta_ci_high'])
print('这些是事后选择的候选市场宇宙内、按记录结算时间筛选的回顾性条件标签结果；不证明真实净利润、钱包技能或可直接实盘部署。')
print('候选宇宙还要求市场在本周开始/结算且成交不越过目录边界，本身用到事后信息；测试集也不代表全部Polymarket。')
print('预测点在记录结算之前仍可能已经公开结果。需要真实结果公开/可交易状态、正确token结算映射、费用、期初成本、转账等数据，以及更长周期和核验后的事件ID。')
if MODE=='smoke':print('SMOKE ONLY：本次仅为运行验证，少量bootstrap且跳过广泛稳健性；不是最终科研结果。')
# Post-smoke distribution diagnostic only: fixed predictions, no refitting/tuning/selection.
# Primary row-weighted scores above remain unchanged; these alternatives were added after the smoke audit.
test_event_sizes=test.event_group.value_counts()
largest_test_event=test_event_sizes.index[0]
event_concentration={'diagnostic_origin':'post-smoke event-distribution audit; exploratory fixed-prediction weighting sensitivity',
    'event_proxy_groups':int(len(test_event_sizes)),
    'effective_event_proxy_groups_inverse_hhi':float(1/((test_event_sizes/test_event_sizes.sum())**2).sum()),
    'largest_test_event':str(largest_test_event),'largest_event_rows':int(test_event_sizes.iloc[0]),
    'largest_event_row_share':float(test_event_sizes.iloc[0]/len(test)),
    'test_wallets_in_multiple_event_proxies':int(test.groupby('wallet').event_group.nunique().gt(1).sum()),
    'bootstrap_limitation':'Event-only resampling does not resample repeated-wallet dependence across event proxies; intervals descriptive and unstable under proxy independence.'}
(OUT/'test_event_concentration_diagnostics.json').write_text(json.dumps(event_concentration,indent=2))
test_event_sizes.rename_axis('event_group').rename('rows').to_csv(OUT/'test_event_proxy_sizes.csv')
weighting_rows=[];drop_largest_mask=test.event_group.ne(largest_test_event).to_numpy()
for name,pred in test_predictions.items():
    yy=test.conditional_profit_label.to_numpy(dtype=float);pp=np.clip(np.asarray(pred),1e-12,1-1e-12)
    individual=pd.DataFrame({'event':test.event_group.to_numpy(),
        'log_loss':-(yy*np.log(pp)+(1-yy)*np.log(1-pp)),'brier':(yy-pp)**2})
    equal=individual.groupby('event')[['log_loss','brier']].mean().mean()
    weighting_rows.append({'model':name,'weighting':'primary_row_weighted_unchanged','rows':len(test),
        'event_proxy_groups':len(test_event_sizes),'log_loss':float(individual.log_loss.mean()),'brier':float(individual.brier.mean())})
    weighting_rows.append({'model':name,'weighting':'posthoc_equal_event_mean','rows':len(test),
        'event_proxy_groups':len(test_event_sizes),'log_loss':float(equal.log_loss),'brier':float(equal.brier)})
    if drop_largest_mask.any():
        alternative_metrics=probability_metrics(test.loc[drop_largest_mask,'conditional_profit_label'],pp[drop_largest_mask])
        weighting_rows.append({'model':name,'weighting':'posthoc_exclude_largest_event',
            'event_proxy_groups':test.loc[drop_largest_mask,'event_group'].nunique(),**alternative_metrics})
weighting_sensitivity=pd.DataFrame(weighting_rows)
weighting_sensitivity.to_csv(OUT/'fixed_prediction_event_weighting_sensitivity.csv',index=False)
weighting_history_delta=[]
for weighting,z in weighting_sensitivity.groupby('weighting',sort=False):
    indexed=z.set_index('model')
    baseline=indexed.loc['current_context_logistic'];full=indexed.loc['current_context_plus_history_logistic']
    weighting_history_delta.append({'weighting':weighting,'log_loss_baseline_minus_history':float(baseline.log_loss-full.log_loss),
        'brier_baseline_minus_history':float(baseline.brier-full.brier),
        'interpretation':'Positive means the frozen full-history model has lower loss; no new CI/p-value or model selection.'})
weighting_history_delta=pd.DataFrame(weighting_history_delta)
weighting_history_delta.to_csv(OUT/'fixed_prediction_weighting_history_deltas.csv',index=False)
assert len(weighting_sensitivity.model.unique())==len(test_predictions)
for row in test_metrics.itertuples(index=False):
    baseline_row=weighting_sensitivity[weighting_sensitivity.model.eq(row.model)&weighting_sensitivity.weighting.eq('primary_row_weighted_unchanged')].iloc[0]
    assert np.isclose(row.log_loss,baseline_row.log_loss,atol=1e-12)
    assert np.isclose(row.brier,baseline_row.brier,atol=1e-12)
print('测试事件集中度/固定预测加权诊断（smoke分布审计后新增，探索性；不重拟合、不选择）：')
display(pd.DataFrame([event_concentration]));display(weighting_sensitivity);display(weighting_history_delta)

# Result-specific Chinese interpretation, generated from executed values rather than a canned conclusion.
from IPython.display import Markdown
ZH_CONCEPTS={'entry_buy_scale':'首次买入金额','entry_price_nonlinear':'买入价格的非线性控制',
 'entry_fill_rows':'首次批次填单数','prior_market_volume':'严格预测前市场成交额',
 'prior_market_participants':'严格预测前市场参与钱包数','observed_market_age':'市场观察年龄',
 'utc_clock':'UTC入场时钟','observation_window_exposure':'观察窗口暴露时长','window_exposure':'观察窗口暴露时长',
 'history_activity':'钱包历史成交活跃度','history_activity_fills':'钱包历史成交活跃度',
 'history_breadth':'钱包历史市场广度','history_breadth_markets':'钱包历史市场广度',
 'history_buy_scale':'钱包历史买入金额','history_buy_concentration':'钱包历史买入市场集中度',
 'history_observed_span':'钱包历史可观察跨度','history_recency':'钱包历史近期程度',
 'missingness':'缺失/无历史状态','entry_direction_rare':'稀疏的首次买入方向集中度'}
lines=['# 本次实际结果解读（'+MODE+'）',
 '所有数值只针对本周事后筛选市场内的条件毛利标签；真实净利润未知。显著性、预测增益和钱包技能是不同概念。',
 f"**测试分布高度集中：{event_concentration['event_proxy_groups']}个事件代理的有效组数仅约{event_concentration['effective_event_proxy_groups_inverse_hhi']:.2f}，最大组占{event_concentration['largest_event_row_share']:.2%}。事件bootstrap未重采样同钱包跨事件依赖；区间基于事件代理独立的近似，具有描述性且可能不稳定，不能作为概念增益的确认性显著证据。**",
 '## 条件关联',
 f"开发训练集 {len(train):,} 行、{train.market.nunique():,} 个市场；两维聚类事件代理 {primary_diag['event_proxy_clusters']} 个，有效事件数（逆HHI）约 {primary_diag['effective_event_clusters_inverse_hhi']:.1f}，最大组占比 {primary_diag['largest_event_share']:.1%}。簇不平衡和启发式事件ID使区间/p值只是近似。"]
for row in primary_tests.itertuples(index=False):
    title=ZH_CONCEPTS.get(row.concept,row.concept)
    if not np.isfinite(row.p_raw):
        lines.append(f'- {title}：本规格无法给出有效联合检验（{row.status}），不能写成“不显著”。');continue
    evidence='在预先规定概念族Holm校正后仍有条件关联证据' if row.p_holm_primary_family<.05 else 'Holm校正后没有清晰的条件关联证据；不等于无效'
    terms=[c for c in row.retained_terms.split(';') if c]
    detail=''
    if len(terms)==1:
        termrow=primary_coef.set_index('term').loc[terms[0]]
        detail=(f"；该项增加一个训练SD的估计差异 {termrow.effect_percentage_points:+.2f} 概率百分点，"
                f"95%区间 [{termrow.ci95_low_percentage_points:+.2f}, {termrow.ci95_high_percentage_points:+.2f}]" )
    else:detail=f'；这是{len(terms)}项联合检验，不能赋予单一线性方向'
    lines.append(f'- {title}：{evidence}（原始p={row.p_raw:.3g}，Holm p={row.p_holm_primary_family:.3g}）{detail}。')
lines.append('历史活跃度与市场广度高度相关；若条件系数方向相反，只表示在另一变量固定时的本规格比较，不能解读为“交易更勤反而能力更差”。价格项与时间项的结果也不证明交易发生时结果尚未知。')
lines+=['## 冻结测试集上的实际预测',f'测试集 {len(test):,} 行、{test.market.nunique():,} 个市场、{test.event_group.nunique()} 个事件代理；训练/测试市场与事件组不重叠。']
for row in test_metrics.itertuples(index=False):
    lines.append(f'- {row.model}：log loss={row.log_loss:.4f}，Brier={row.brier:.4f}，ROC-AUC={row.roc_auc:.4f}，PR-AUC={row.pr_auc_average_precision:.4f}。')
lo,hi=history_delta['log_loss_delta_ci_low'],history_delta['log_loss_delta_ci_high']
if lo>0:history_reading='加入整组钱包历史后，这个固定logistic模型在本测试事件上的log loss有改善证据。'
elif hi<0:history_reading='这个固定完整logistic模型没有展现历史组的增益；删去历史反而降低本次log loss。这不证明历史信息本身有害。'
else:history_reading='整组钱包历史的log loss变化区间跨0，本次测试没有清晰的增益证据，也不能证明它没有预测价值。'
lines.append(f"基线−加入历史的Δlog loss={history_delta['log_loss_delta']:+.5f}，事件bootstrap95%区间 [{lo:+.5f}, {hi:+.5f}]。"+history_reading)
lines+=['## 各概念的预测贡献（固定测试集探索）','正Δ=删去概念后更差；高相关特征可相互替代。这些固定比较没有反过来用于模型/变量选择。']
for row in ablations.itertuples(index=False):
    if row.log_loss_delta_ci_low>0:reading='删去后更差，本规格显示条件增益'
    elif row.log_loss_delta_ci_high<0:reading='删去后更低损失，完整规格未显示增益；不证明信息有害'
    else:reading='区间跨0，未显示清晰独立增益；不等于没有价值'
    lines.append(f"- {ZH_CONCEPTS.get(row.dropped_concept,row.dropped_concept)}：Δlog loss={row.log_loss_delta:+.5f}，95%区间 [{row.log_loss_delta_ci_low:+.5f}, {row.log_loss_delta_ci_high:+.5f}]；{reading}。")
lines+=['## 事件集中度的事后分布敏感性（固定全部模型预测）',
 '这是smoke分布审计后新增的探索性读数：不改变主要逐行加权分数，不重拟合、不调参，也不选择或排除模型。等权事件均值给予每个事件相同权重；去掉最大事件改变了测试人群，不是宣布那个事件错误。这里没有新增区间或p值。']
for row in weighting_sensitivity.itertuples(index=False):
    if row.weighting!='primary_row_weighted_unchanged':
        lines.append(f'- {row.weighting} / {row.model}：log loss={row.log_loss:.4f}，Brier={row.brier:.4f}。')
for row in weighting_history_delta.itertuples(index=False):
    lines.append(f'- {row.weighting}：基线−历史Δlog loss={row.log_loss_baseline_minus_history:+.5f}，ΔBrier={row.brier_baseline_minus_history:+.5f}。只描述固定预测对权重/人群的敏感性。')
lines+=['## 限制与改善方向',
 '反转结果映射、去重假设、价格中段与重复受影响行排除的结果另见敏感性表；它们改变标签或人群，不能由稳定性反推真实净利润已经验证。',
 '候选宇宙用到了周内开始/结算和目录时间覆盖等事后条件；测试只针对观察到的晚入场者。预测早于记录结算不等于早于结果公开。需要真实结果公开/交易状态、正确结果-token映射、费用/期初成本/非交易持仓变化、核验事件分组及更多日期，才能加强实际部署和总体推断。']
if MODE=='smoke':lines.insert(1,'**本次为smoke运行验证，bootstrap次数少且跳过广泛稳健性，不作为最终科研结论。**')
interpretation_zh='\n\n'.join(lines)
(OUT/'interpretation_zh.md').write_text(interpretation_zh)
display(Markdown(interpretation_zh))

provenance.update({'cohort_summary':cohort_summary.to_dict('records'),'primary_diagnostics':primary_diag,
    'test_event_concentration':event_concentration,'full_source_feature_count':49,'event_rules':events.event_mapping_rule.value_counts().to_dict(),
    'split_assignment_uses_labels':False,'threshold_selected_on_test':False,'fixed_hyperparameters':True,
    'elapsed_seconds':time.perf_counter()-START,'output_hashes':{p.name:sha(p) for p in sorted(OUT.glob('*.csv'))}})
Path(OUT/'modeling_provenance.json').write_text(json.dumps(provenance,indent=2,default=str))
print('全部模型阶段完成；耗时秒:',round(time.perf_counter()-START,2),'；输出目录:',str(OUT))
